# Create a Q&A Chatbot with LangChain Project

### Set the OpenAI API Key as an Environment Variable

In [4]:
%load_ext dotenv
%dotenv

### Import the Libraries

In [3]:
from langchain_chroma.vectorstores import Chroma
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers.string import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter, TokenTextSplitter

### Load the Course Transcript

In [5]:
loader_pdf = PyPDFLoader("./Introduction_to_Tableau.pdf")

In [6]:
docs_list = loader_pdf.load()

In [ ]:
print(docs_list[1])

In [8]:
string_list_concat = "".join(doc.page_content for doc in docs_list)

### Split the Course Transcript with MarkdownHeaderTextSplitter

In [9]:
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "Header 1"), ("##", "Header 2")]
)

In [10]:
docs_list_md_split = md_splitter.split_text(string_list_concat)

In [ ]:
print(docs_list_md_split[1])

### Create a Chain to Correct the Course Transcript

In [11]:
string_list_split = [doc.page_content for doc in docs_list_md_split]

In [ ]:
print(string_list_split[1])

In [13]:
PROMPT_FORMATTING_S = """Improve the following Tableau lecture transcript by:
- Splitting the text into meaningful paragraphs
- Correcting any misplaced punctuation
- Fixing mistranscribed words (e.g., changing 'tableaux' to 'Tableau')"
"""

PROMPT_TEMPLATE_FORMATTING_H = """This is the transcript:
{lecture_transcript}
"""

In [14]:
prompt_formatting_s = SystemMessage(content=PROMPT_FORMATTING_S)
prompt_template_formatting_h = HumanMessagePromptTemplate.from_template(
    PROMPT_TEMPLATE_FORMATTING_H
)
chat_prompt_template_formatting = ChatPromptTemplate(
    messages=[prompt_formatting_s, prompt_template_formatting_h]
)

In [19]:
chat = ChatOpenAI(model="gpt-4o", seed=365, temperature=0)

In [16]:
str_output_parser = StrOutputParser()

In [21]:
chain_formatting = chat_prompt_template_formatting | chat | str_output_parser

In [ ]:
string_list_formatted = chain_formatting.batch(string_list_split)

In [ ]:
print(string_list_formatted[1])

In [ ]:
# Override the docs_list_md_split list such that the page_content parameter of each Document objects stores the updated lecture.


In [ ]:
for doc, formatted_lecture in zip(docs_list_md_split, string_list_formatted, strict=False):
    doc.page_content = formatted_lecture

### Split the Lectures with TokenTextSplitter

In [ ]:
token_splitter = TokenTextSplitter(chunk_size=500, chunk_overlap=50)

In [ ]:
docs_list_tokens_split = token_splitter.split_documents(docs_list_md_split)

### Create Embeddings, Vector Store, and Retriever

In [ ]:
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
vectorstore = Chroma.from_documents(documents=docs_list_tokens_split, embedding=embedding)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

### Create Prompts and Prompt Templates for the Q&A Chatbot Chain

In [ ]:
PROMPT_CREATING_QUESTION = """Lecture: {question_lecture}
Title: {question_title}
Body: {question_body}"""

PROMPT_RETRIEVING_S = """You will receive a question from a student taking a Tableau course, which includes a title and a body.
The corresponding lecture will also be provided.

Answer the question using only the provided context.

At the end of your response, include the section and lecture names where the context was drawn from, formatted as follows:
Resources:
Section: *Section Title*, Lecture: *Lecture Title*
...
Replace *Section Title* and *Lecture Title* with the appropriate titles."""

PROMPT_TEMPLATE_RETRIEVING_H = """This is the question:
{question}

This is the context:
{context}"""

prompt_creating_question = PromptTemplate.from_template(PROMPT_CREATING_QUESTION)
prompt_retrieving_s = SystemMessage(content=PROMPT_RETRIEVING_S)
prompt_template_retrieving_h = HumanMessagePromptTemplate.from_template(
    PROMPT_TEMPLATE_RETRIEVING_H
)

chat_prompt_template_retrieving = ChatPromptTemplate(
    messages=[prompt_retrieving_s, prompt_template_retrieving_h]
)

### Create the First Version of the Q&A Chatbot Chain

In [ ]:
chain_retrieving = (
    {"question": prompt_creating_question, "context": prompt_creating_question | retriever}
    | chat_prompt_template_retrieving
    | chat
    | str_output_parser
)

In [ ]:
result = chain_retrieving.invoke(
    {
        "question_lecture": "Adding a custom calculation",
        "question_title": "Why are we using SUM here? It's unclear to me.",
        "question_body": "This question refers to calculating the GM%.",
    }
)

In [ ]:
result

### Create a Runnable Function to Format the Context

In [ ]:
def format_context(dictionary):
    documents = retriever.invoke(dictionary["question"])

    context_list = []
    for doc in documents:
        section_title = doc.metadata.get("Header 1", "Unknown Section")
        lecture_title = doc.metadata.get("Header 2", "Unknown Lecture")
        context_list.append(
            f"Section: {section_title}\nLecture: {lecture_title}\nContent: {doc.page_content}"
        )

    return "\n\n".join(context_list)

In [ ]:
chain_retrieving_improved = (
    {
        "question": prompt_creating_question,
        "context": prompt_creating_question | RunnableLambda(format_context),
    }
    | chat_prompt_template_retrieving
    | chat
    | str_output_parser
)

In [ ]:
result_improved = chain_retrieving_improved.invoke(
    {
        "question_lecture": "Adding a custom calculation",
        "question_title": "Why are we using SUM here? It's unclear to me.",
        "question_body": "This question refers to calculating the GM%.",
    }
)

In [ ]:
result_improved

### Stream the Response

In [ ]:
result_streamed = chain_retrieving_improved.stream(
    {
        "question_lecture": "Adding a custom calculation",
        "question_title": "Why are we using SUM here? It's unclear to me.",
        "question_body": "This question refers to calculating the GM%.",
    }
)

In [ ]:
# Create a for-loop to stream the response


In [ ]:
for chunk in result_streamed:
    print(chunk, end="", flush=True)